# 02 — Baseline TF-IDF + Logistic Regression

**Tujuan:** membuat pembanding sederhana sebelum menyimpulkan IndoBERT, dengan hyperparameter dan threshold dari validation.

**Prasyarat:** `data/processed/{train,val,test}.csv` sudah ada.

**Estimasi waktu:** 1–3 menit.

## Glosarium

- **TF-IDF**: bobot kata/ngram yang khas untuk sebuah teks.
- **N-gram**: urutan 1–2 kata, misalnya "dasar bodoh".
- **Logistic Regression**: model linear yang mengeluarkan probabilitas.
- **Class weight balanced**: memberi bobot lebih pada kelas minoritas.
- **Threshold**: ambang probabilitas untuk memutuskan bully.

## 1. Muat data

Kita membaca split hasil pipeline. Teks sudah dinormalisasi ringan, dengan emoji
dan tanda baca dipertahankan. Baseline dan IndoBERT nanti memakai bahan yang
sama, sehingga perbandingannya adil. Tidak ada fitting pada validation.

In [ ]:
import json

import pandas as pd

from cyberbully.baseline import train_baseline
from cyberbully.config import DATA_PROCESSED, LABEL_COLUMN
from cyberbully.data import read_split

train_df = read_split(DATA_PROCESSED, "train")
val_df = read_split(DATA_PROCESSED, "val")
print("train:", train_df.shape, "| val:", val_df.shape)
print("positif train:", int(train_df[LABEL_COLUMN].sum()), "| positif val:", int(val_df[LABEL_COLUMN].sum()))

## 2. Latih baseline

`train_baseline` mencoba beberapa nilai `C` (regularisasi), memilih F1 kelas bully
terbaik pada validation di threshold 0,5, lalu memilih threshold yang memberi F1
terbaik. Semua keputusan memakai validation; test tidak disentuh.

In [ ]:
pipeline, payload, features = train_baseline(train_df, val_df)
print("C terpilih:", payload["best_c"])
print("threshold:", round(float(payload["threshold"]), 2))
print(json.dumps(payload["val"], indent=2))

## 3. Membaca metrik

Accuracy saja bisa menyesatkan saat kelas tidak seimbang. Precision menjawab
"berapa persen prediksi bully yang benar", recall menjawab "berapa persen bully
yang tertangkap", dan F1 menyeimbangkan keduanya. Confusion matrix menunjukkan
false positive (aman dituduh bully) dan false negative (bully lolos).

In [ ]:
metrics = payload["val"]
table = pd.DataFrame(
    {
        "metrik": ["accuracy", "precision bully", "recall bully", "F1 bully"],
        "nilai": [metrics["accuracy"], metrics["precision_bully"], metrics["recall_bully"], metrics["f1_bully"]],
    }
)
print(table.round(3).to_string(index=False))
print("confusion matrix [[TN, FP], [FN, TP]]:", metrics["confusion_matrix"])

## 4. Fitur paling berpengaruh

Koefisien Logistic Regression menunjukkan ngram yang mendorong prediksi ke kelas
bully atau non-bully. Ini bukan bukti kausal dan bisa mencerminkan bias anotasi.
Gunakan untuk sanity check: apakah model belajar kata yang masuk akal, atau
justru kata netral yang kebetulan sering muncul di data bully?

In [ ]:
print("Pendorong bully:", ", ".join(features["bully"][:15]))
print("Pendorong non-bully:", ", ".join(features["non_bully"][:15]))

## 5. Batas baseline

TF-IDF tidak memahami urutan kata, negasi rumit, atau sindiran tanpa kata kasar.
Namun baseline yang kuat penting: kalau IndoBERT hanya sedikit lebih baik,
masalahnya mungkin di data atau label, bukan arsitektur. Angka baseline dipakai
sebagai pembanding di notebook `04`.

## Ringkasan

- Baseline dilatih pada train; `C` dan threshold dipilih dari validation.
- Metrik utama adalah F1 bully, disertai precision, recall, dan confusion matrix.
- Fitur teratas hanya untuk pemeriksaan, bukan klaim kausal.
- Test set belum disentuh; evaluasi test dilakukan sekali di akhir.